# Local ROM Stability Check 2D -- Cahn-Hilliard


Build a FOM trajectory from a chosen initial condition, construct Petrov-Galerkin/POD/DEIM ROMs, run the true-nonlinearity ROM, and inspect capture/trajectory/modes before training a neural ROM.

In [ ]:
using Random
using Plots

### ADJUSTED: Use shared ROM stability helpers instead of notebook-local function definitions.
include(joinpath(@__DIR__, "..", "..", "src", "HPC", "Tools", "ROM_stability_helpers.jl"))
include(joinpath(@__DIR__, "..", "..", "src", "Visualizations", "optimization_visualizations.jl"))


## Parameters

In [ ]:
N = 128
L = 4.0*pi
ε2 = .01
sigma = 1.0
mean_c = .0
tspan = (0.0, 20.0)
reference_dt_factor = 0.5
dimension = 2
boundary_condition = "periodic"
N_obs = 100
h = 8
seed = 1



initial_condition_examples = [
    (name="default", u0=nothing),
    (name="2d cosine product", u0=(x, y) -> mean_c + 0.10cos(4π * x / L) * cos(4π * y / L)),
    (name="2d low frequency xy", u0=(x, y) -> mean_c + 0.10sin(2π * x / L) * sin(2π * y / L)),
    (name="2d high frequency x", u0=(x, y) -> mean_c + 0.05cos(8π * x / L)),
    (name="2d offcenter bump", u0=(x, y) -> mean_c + 0.12exp(-(((x - 0.35L)^2 + (y - 0.58L)^2) / (2 * (0.08L)^2)))),
    (name="2d mixed modes", u0=(x, y) -> mean_c + 0.06sin(6π * x / L) * sin(4π * y / L) + 0.04cos(2π * x / L + 0.7) * cos(6π * y / L)),
    (name="2d random scalar field", u0=begin
        rng = MersenneTwister(seed)
        values = randn(rng, N, N)
        values .-= mean(values)
        values .= mean_c .+ 0.10 .* values ./ maximum(abs.(values))
        values
    end),
];
initial_condition_2d_examples = filter(ic -> startswith(ic.name, "2d "), initial_condition_examples)


## Functions

## Initial Conditions

In [ ]:
for k in 1:length(initial_condition_2d_examples)
    # println("k=$k")
    selected_initial_condition_2d_index = k
    selected_initial_condition = initial_condition_2d_examples[selected_initial_condition_2d_index]
    display(
        ch_plot_stability_initial_conditions([selected_initial_condition];
         N, L, ε2, dimension, boundary_condition, show_colorbar = true)
         )
end

## Run FOM, Build ROM, Run ROM

In [ ]:
### ADJUSTED: Use a richer C-H initial condition so the requested ROM rank is available.
selected_initial_condition = initial_condition_2d_examples[end]

u0 = selected_initial_condition.u0
fom = ch_run_stability_fom_reference(; N, L, ε2, sigma, mean_c, tspan, reference_dt_factor, u0, dimension, boundary_condition)
println("Ran FOM reference")


In [ ]:
r=100
m=100
rom = ch_build_stability_rom(fom, r, m; N_obs, h, seed)
println("Built ROM")
rom_run = ch_run_stability_rom(fom, rom);
println("Ran ROM ")

## Singular-Value Capture

In [ ]:
stability_capture_table(fom, ch_build_stability_rom; rs=[2, 4, 8, 10, 15, 20], ms=[2, 4, 8, 10, 15, 20])

## Trajectory

In [ ]:
trajectory_gifs(fom, rom_run; out_dir=joinpath(@__DIR__, "rom_stability_gifs"), max_frames=150, fps=15, show_colorbar = false)

In [ ]:
true_u = hcat(fom.u_ref.u...)
rom_u = rom_run.reconstructed

true_change = [maximum(abs.(true_u[:, j] .- true_u[:, 1])) for j in axes(true_u, 2)]
rom_change = [maximum(abs.(rom_u[:, j] .- rom_u[:, 1])) for j in axes(rom_u, 2)]

@show size(true_u)
@show size(rom_u)
@show length(fom.u_ref.t)
@show extrema(true_u)
@show extrema(rom_u)
@show maximum(true_change)
@show maximum(rom_change)
@show true_change[1:10:end]

## Modes

In [ ]:
plot_rom_modes(fom, rom, n_state=10, n_deim=10)